<h1>Producer</h1>

## Importing Libraries

In [2]:
import pandas as pd

<hr>

## Getting the data

In [3]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [7]:
## Picking one single event
row = df.iloc[0]
print(row)

PULocationID                             43
DOLocationID                            186
trip_distance                          1.68
total_amount                          22.15
tpep_pickup_datetime    2025-11-01 00:13:25
Name: 0, dtype: object


In [11]:
row.tpep_pickup_datetime.timestamp()

1761956005.0

### Creating our own class to process rides

In [6]:
from dataclasses import dataclass

In [8]:
@dataclass
class Ride:
    #defining the fields of the dataclass
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int #epoch milliseconds

In [12]:
ride = Ride(
    PULocationID = 1,
    DOLocationID = 10,
    trip_distance = 10,
    total_amount = 20,
    tpep_pickup_datetime = 1761956005000
)

## Setting up the streaming process

In [14]:
import json
from kafka import KafkaProducer

In [13]:
# transforming data into string format to send to kafka
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

In [15]:
#creating kafka producer instance
server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer #turn python objects into json strings
)

In [16]:
import dataclasses

topic_name = 'rides' #defining topic

producer.send(topic_name, value=dataclasses.asdict(ride)) #sending topic (data) to kafka
producer.flush()